# Forcing a shape

**Scenario:** a claims intake service reads first notice of loss notes and files them. The system
prompt says "reply with JSON only, no prose, no markdown". Review passed. It ran for a week.

Then a morning of notes filed nothing at all, and the queue backed up behind them.

Asking for a shape is a request. Getting one is a setting, and the two live in different parts of the
call. Think of it as a form with numbered boxes rather than a blank sheet. A blank sheet invites a
covering letter.

## Mechanics

There are four ways to ask a model for data, and they do not make the same promise.

| How you ask | What you send | What comes back |
|---|---|---|
| In words | an instruction in the system prompt | `message.content`, a string formatted however the model liked |
| `tools` alone | one schema per tool | maybe a tool call, maybe prose. The model picks |
| `tools` plus `tool_choice` | the same schema, plus one tool name | `message.tool_calls`, every time, with `arguments` as JSON text |
| `response_format` | a schema and no tool | `message.content`, held to that schema |

A schema is the written shape of the allowed data, including types and required fields. The setting
that forces the model through a named tool is `tool_choice`. Which of these a model supports is a
provider fact, so ask.

In [1]:
from vault import load_env, model_for, provider_truth

load_env()
model = model_for("default")
supported = provider_truth()["models"][model]["supported_parameters"]

for setting in ("tools", "tool_choice", "response_format", "structured_outputs"):
    print(f"  {setting:20} {setting in supported}")

  tools                True
  tool_choice          True
  response_format      True
  structured_outputs   True


## The picture

![Two ways to ask, and only one of them constrains the answer](images/forcing-a-shape.svg)

Both paths reach the same model. Only one of them comes back in a shape a parser can rely on.

## The cost

```
cost = notes that failed to parse x (one more call + somebody re-keying the note by hand)
```

A parse failure is paid twice. Once for the retry, and once for the time it takes a person to type
the note in again.

## The failure

Four intake notes, written the way a call handler writes them. The instruction asks for JSON only, in
the plainest words available.

In [2]:
NOTES = [
    "Claim FNOL-2291. Ana Ruiz, storm damage to the roof at 12 Harbour Lane on 3 March. About 8400 dollars. Nobody hurt.",
    "FNOL-2292: caller was upset. Kitchen flood, burst pipe, maybe 15000? He also asked about his other policy.",
    "FNOL-2293 rear-end collision on the M4, driver has whiplash, garage quote 3100 GBP.",
    "FNOL-2294 - theft of bicycle from garage, no receipt, guessing 900.",
]

SYSTEM = ("You are a claims intake assistant. Reply with JSON only, no prose, no markdown. "
          'Shape: {"claim_id": str, "peril": str, "estimate_usd": number, "injuries": bool}')

The parser is the ordinary one. It reads the content and hands it to `json.loads`.

In [3]:
import json
from vault import get_client

client = get_client("08-deterministic-outputs/01-forcing-a-shape")


def ask_in_words(note):
    """Ask for JSON with an instruction, and parse whatever comes back."""
    reply = client.chat.completions.create(
        model=model, max_tokens=300,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": note}])
    return reply.choices[0].message.content

Run all four and count how many the queue could file.

In [4]:
filed = []
for note in NOTES:
    text = ask_in_words(note)
    try:
        filed.append(json.loads(text))
    except json.JSONDecodeError as exc:
        print(f"  unparsed: {exc.msg}, first characters {text[:12]!r}")

print(f"\nfiled {len(filed)} of {len(NOTES)} notes")
assert len(filed) == len(NOTES), f"{len(NOTES) - len(filed)} notes could not be filed"

  unparsed: Expecting value, first characters '```json\n{\n  '
  unparsed: Expecting value, first characters '```json\n{\n  '
  unparsed: Expecting value, first characters '```json\n{\n  '
  unparsed: Expecting value, first characters '```json\n{\n  '

filed 0 of 4 notes


AssertionError: 4 notes could not be filed

## The diagnosis

Four notes, four failures, and the content was right every time. The model wrapped each answer in a
markdown code fence, so the first character the parser met was a backtick.

The mechanic is the first row of the table. An instruction in the system prompt rides in the same
channel as the answer, so nothing in the request made the reply JSON. The model read "no markdown" as
a preference.

Stripping the fence looks like the fix. It is a guess about one habit of one model, and the next
release has a different habit.

## The fix

Send the same shape as a schema, give it a name, and force the model through it. The format stops
being a question anyone has to guess at.

In [5]:
FILE_CLAIM = {
    "type": "function",
    "function": {
        "name": "file_claim",
        "description": "File a first notice of loss.",
        "parameters": {
            "type": "object",
            "properties": {"claim_id": {"type": "string"},
                           "peril": {"type": "string"},
                           "estimate_usd": {"type": "number"},
                           "injuries": {"type": "boolean"}},
            "required": ["claim_id", "peril", "estimate_usd", "injuries"],
            "additionalProperties": False,
        },
    },
}

The request builder is its own function, so the forcing lives in one place and a test can read it
without calling anything.

In [6]:
def build_request(note):
    """One place decides how the model is allowed to answer."""
    return {"model": model, "max_tokens": 300, "tools": [FILE_CLAIM],
            "tool_choice": {"type": "function", "function": {"name": "file_claim"}},
            "messages": [{"role": "system", "content": "You are a claims intake assistant."},
                         {"role": "user", "content": note}]}

The arguments still arrive as text, so they still need parsing. The difference is that they are JSON
by construction rather than by request.

In [7]:
def file_claim(note):
    """Forced through the named tool. Returns the parsed arguments."""
    reply = client.chat.completions.create(**build_request(note))
    call = reply.choices[0].message.tool_calls[0]
    return json.loads(call.function.arguments)

Same four notes, same schema, different setting.

In [8]:
forced = [file_claim(note) for note in NOTES]

print(f"in words        : {len(filed)} of {len(NOTES)} notes filed")
print(f"forced by schema: {len(forced)} of {len(NOTES)} notes filed\n")
for claim in forced:
    print(f"  {claim['claim_id']}  peril={claim['peril']!r:20} {claim['estimate_usd']}")

in words        : 0 of 4 notes filed
forced by schema: 4 of 4 notes filed

  FNOL-2291  peril='wind'               8400
  FNOL-2292  peril='water'              15000
  FNOL-2293  peril='Collision'          3100
  FNOL-2294  peril='theft'              900


## The gate

The regression to guard against is somebody deleting `tool_choice` during a tidy up, because the
model still answers correctly most of the time without it. This test reads the request rather than
sending it, so it needs no key and runs in milliseconds.

In [9]:
def test_the_request_always_forces_the_tool():
    request = build_request("any note")
    choice = request.get("tool_choice")
    assert choice, "tool_choice is gone, the model may answer in prose again"
    assert choice["function"]["name"] == FILE_CLAIM["function"]["name"]
    assert FILE_CLAIM["function"]["parameters"]["additionalProperties"] is False


test_the_request_always_forces_the_tool()
print("gate holds: every claim request names the tool it must answer through")

gate holds: every claim request names the tool it must answer through


Delete the `tool_choice` line from `build_request` and this test fails.

Every note parses now, and one field still cannot be trusted. Look at `peril`. Storm damage came back
as `wind`, and a collision came back capitalised, which no routing table will match. Forcing a reply
into a fixed shape fixed the parsing and nothing else. That gap is the next sub-module.

### Enterprise exploration

- Intake runs at hundreds of notes an hour, and thousands after a storm. Which of these four ways to
  ask survives that, and what does a retry cost in latency?
- A forced tool call bills for the schema on every request. What does that add per million notes?
- The regulator asks how a claim value was derived. What do you keep, and for how long?
- The provider changes what `tool_choice` accepts. How do you find out before your customers do?

### Key takeaways

- An instruction about format travels in the same channel as the answer, so nothing enforces it.
- `tool_choice` naming one tool turns the shape from a request into a setting.
- `arguments` is still text. Parse it, and keep the parse in one place.
- A shape you can rely on is not an answer you can rely on.